In [1]:
import sqlite3
import pandas as pd
import numpy as np

# 1. Load existing cleaned dataset
df_app = pd.read_csv('applicants_cleaned.csv')

# Add applicant_id column if not present
if 'applicant_id' not in df_app.columns:
    df_app['applicant_id'] = range(1, len(df_app) + 1)

# Rename columns to match SQL database standards
df_app_db = df_app.rename(columns={
    'Applicant Name': 'name',
    'Email': 'email',
    'Phone': 'phone',
    'Domain Applied': 'domain',
    'University': 'university',
    'Application Date': 'application_date',
    'Status': 'status'
})[['applicant_id', 'name', 'email', 'domain', 'university', 'application_date', 'status']]

# 2. Generate 'interns' table from 'Selected' applicants
selected_apps = df_app_db[df_app_db['status'] == 'Selected'].copy()

# Fallback if selected count is very low
if len(selected_apps) < 10:
    selected_apps = df_app_db.head(20).copy()

np.random.seed(42)
interns_list = []

for idx, row in selected_apps.iterrows():
    intern_id = len(interns_list) + 1
    completion_status = np.random.choice(['Completed', 'Dropped Out'], p=[0.75, 0.25])
    interns_list.append({
        'intern_id': intern_id,
        'applicant_id': row['applicant_id'],
        'domain': row['domain'],
        'start_date': '2026-06-01',
        'completion_status': completion_status
    })

df_interns = pd.DataFrame(interns_list)

# 3. Generate 'hackathon_scores' table for Completed interns
completed_interns = df_interns[df_interns['completion_status'] == 'Completed']
scores_list = []

for idx, row in completed_interns.iterrows():
    score = int(np.random.normal(82, 10))
    score = min(max(score, 55), 98) # Keep scores realistic between 55 and 98
    scores_list.append({
        'intern_id': row['intern_id'],
        'score': score,
        'domain': row['domain']
    })

df_scores = pd.DataFrame(scores_list)

# 4. Save individual CSV files
df_app_db.to_csv('applicants.csv', index=False)
df_interns.to_csv('interns.csv', index=False)
df_scores.to_csv('hackathon_scores.csv', index=False)

# 5. Save all tables into nextgen.db SQLite database
conn = sqlite3.connect('nextgen.db')
df_app_db.to_sql('applicants', conn, if_exists='replace', index=False)
df_interns.to_sql('interns', conn, if_exists='replace', index=False)
df_scores.to_sql('hackathon_scores', conn, if_exists='replace', index=False)
conn.close()

print("Step 1 Complete: Created nextgen.db, applicants.csv, interns.csv, and hackathon_scores.csv!")

Step 1 Complete: Created nextgen.db, applicants.csv, interns.csv, and hackathon_scores.csv!


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nextgen.db')

# Sanity Check - Displaying first 5 rows of each table
print("Applicants Table:")
display(pd.read_sql("SELECT * FROM applicants LIMIT 5;", conn))

print("Interns Table:")
display(pd.read_sql("SELECT * FROM interns LIMIT 5;", conn))

print("Hackathon Scores Table:")
display(pd.read_sql("SELECT * FROM hackathon_scores LIMIT 5;", conn))

Applicants Table:


,applicant_id,name,email,domain,university,application_date,status
0,1,Ali Farooq,ali.farooq1@example.com,App Development,Government College University Faisalabad,2026-08-08,Selected
1,2,Saad Farooq,saad.farooq2@example.com,None,COMSATS University,None,Under Review
2,3,Noor Ali,noor.ali3@example.com,Graphic Design,The University of Faisalabad,None,Under Review
3,4,LAIBA SIDDIQUI,laiba.siddiqui4@example.com,Digital Marketing,The University of Faisalabad,2026-10-12,Under Review
4,5,Iqra Farooq,iqra.farooq5@example.com,Graphic Design,COMSATS University,None,Under Review


Interns Table:


,intern_id,applicant_id,domain,start_date,completion_status
0,1,1,App Development,2026-06-01,Completed
1,2,7,Data Science,2026-06-01,Dropped Out
2,3,10,Graphic Design,2026-06-01,Completed
3,4,11,Graphic Design,2026-06-01,Completed
4,5,18,Digital Marketing,2026-06-01,Completed


Hackathon Scores Table:


,intern_id,score,domain
0,1,74,App Development
1,3,60,Graphic Design
2,4,75,Graphic Design
3,5,87,Digital Marketing
4,6,98,Digital Marketing


# Query 1: How many interns completed each domain's program?

In [5]:
query_1 = """
SELECT domain, COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
"""
pd.read_sql(query_1, conn)

,domain,completed_count
0,Graphic Design,5
1,Digital Marketing,5
2,None,3
3,Web Development,2
4,Data Science,2
5,App Development,2


# Query 2: What is the average hackathon score per domain?

In [6]:
query_2 = """
SELECT domain, ROUND(AVG(score), 2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""
pd.read_sql(query_2, conn)

,domain,avg_score
0,Data Science,83.5
1,None,83.0
2,Digital Marketing,81.8
3,Web Development,79.0
4,Graphic Design,75.2
5,App Development,72.5


# Query 3: Which interns scored above a threshold (top performers)?

In [7]:
query_3 = """
SELECT i.intern_id, i.domain, h.score
FROM interns i
JOIN hackathon_scores h ON i.intern_id = h.intern_id
WHERE h.score >= 85
ORDER BY h.score DESC;
"""
pd.read_sql(query_3, conn)

,intern_id,domain,score
0,6,Digital Marketing,98
1,14,Data Science,91
2,5,Digital Marketing,87
3,7,Graphic Design,85
4,22,None,85


# Query 4: Conversion rate from "applied" to "completed" per domain

In [8]:
query_4 = """
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id) / COUNT(DISTINCT a.applicant_id),
        2
    ) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
    ON a.applicant_id = i.applicant_id AND i.completion_status = 'Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
"""
pd.read_sql(query_4, conn)

,domain,total_applicants,total_completed,conversion_rate_pct
0,Digital Marketing,15,5,33.33
1,Graphic Design,16,5,31.25
2,App Development,8,2,25.00
3,None,13,3,23.08
4,Web Development,9,2,22.22
5,Data Science,12,2,16.67


# Query 5: Average hackathon score by university for completed interns

In [9]:
query_5 = """
SELECT
    a.university,
    COUNT(DISTINCT i.intern_id) AS total_interns,
    ROUND(AVG(h.score), 2) AS avg_score
FROM applicants a
JOIN interns i ON a.applicant_id = i.applicant_id
JOIN hackathon_scores h ON i.intern_id = h.intern_id
GROUP BY a.university
ORDER BY avg_score DESC;
"""
pd.read_sql(query_5, conn)

,university,total_interns,avg_score
0,National University of Sciences and Technology,3,84.00
1,University of Lahore,1,83.00
2,Punjab University,5,82.80
3,COMSATS University,4,81.50
4,Government College University Faisalabad,1,74.00
5,University of Sargodha,4,73.75
6,The University of Faisalabad,1,60.00
